In [1]:
# Environment Setup and Folder Paths
from pathlib import Path

PROJECT_ROOT = Path(r"E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 5\Task5-1_ObjectDetection_Segmentation")
DATASET_DIR = PROJECT_ROOT / "dataset"
TEST_IMAGES_DIR = DATASET_DIR / "test_images"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
SCREENSHOTS_DIR = PROJECT_ROOT / "screenshots"
REPORT_DIR = PROJECT_ROOT / "report"

for d in [DATASET_DIR, TEST_IMAGES_DIR, OUTPUTS_DIR, SCREENSHOTS_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TEST_IMAGES_DIR:", TEST_IMAGES_DIR)
print("OUTPUTS_DIR:", OUTPUTS_DIR)
print("SCREENSHOTS_DIR:", SCREENSHOTS_DIR)
print("REPORT_DIR:", REPORT_DIR)

PROJECT_ROOT: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 5\Task5-1_ObjectDetection_Segmentation
TEST_IMAGES_DIR: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 5\Task5-1_ObjectDetection_Segmentation\dataset\test_images
OUTPUTS_DIR: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 5\Task5-1_ObjectDetection_Segmentation\outputs
SCREENSHOTS_DIR: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 5\Task5-1_ObjectDetection_Segmentation\screenshots
REPORT_DIR: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 5\Task5-1_ObjectDetection_Segmentation\report


In [2]:
# Verify Core Libraries
import sys
import cv2
import numpy as np
import tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt

print("Python Path:", sys.executable)
print("cv2:", cv2.__version__)
print("numpy:", np.__version__)
print("tf:", tf.__version__)

Python Path: e:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 5\tf_env\Scripts\python.exe
cv2: 4.10.0
numpy: 1.26.4
tf: 2.20.0


In [3]:
# Check Test Images Folder
from pathlib import Path

images = sorted(list(TEST_IMAGES_DIR.glob("*.*")))
print("Images found:", len(images))
for p in images[:15]:
    print("-", p.name)

if len(images) < 10:
    print("\n❗ Please copy 10 images into:")
    print(TEST_IMAGES_DIR)
    print("Rename them like: img01.jpg ... img10.jpg (recommended)")

Images found: 10
- image1.jpg
- image10.jpg
- image2.jpg
- image3.jpg
- image4.jpg
- image5.jpg
- image6.jpg
- image7.jpg
- image8.jpg
- image9.jpg


In [4]:
# Display 10 Test Images (Quick Visual Check)
def show_images_grid(paths, cols=5, figsize=(14, 6)):
    paths = paths[:10]
    rows = (len(paths) + cols - 1) // cols
    plt.figure(figsize=figsize)
    for i, p in enumerate(paths):
        img = Image.open(p).convert("RGB")
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(p.name, fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

images = sorted(list(TEST_IMAGES_DIR.glob("*.*")))
show_images_grid(images, cols=5)

<Figure size 1400x600 with 10 Axes>

In [5]:
# Load YOLO Model (COCO Pretrained)
from ultralytics import YOLO

yolo_model = YOLO("yolov8n.pt")  # small + fast
print("YOLO model loaded:", yolo_model.model.yaml.get("name", "yolov8"))

YOLO model loaded: yolov8


In [6]:
# Run YOLO Inference on 10 Images and Save Visual Outputs
import time

yolo_out_dir = OUTPUTS_DIR / "det_yolo_v8n"
yolo_out_dir.mkdir(exist_ok=True)

images = sorted(list(TEST_IMAGES_DIR.glob("*.*")))[:10]

times = []
for p in images:
    t0 = time.time()
    results = yolo_model.predict(source=str(p), conf=0.25, verbose=False)
    t1 = time.time()
    times.append(t1 - t0)

    # Save annotated image
    annotated = results[0].plot()
    out_path = yolo_out_dir / p.name
    cv2.imwrite(str(out_path), annotated)

print("Saved YOLO outputs to:", yolo_out_dir)
print("Avg inference time (sec/image):", round(float(np.mean(times)), 4))

Saved YOLO outputs to: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 5\Task5-1_ObjectDetection_Segmentation\outputs\det_yolo_v8n
Avg inference time (sec/image): 0.195


In [7]:
# Preview YOLO Outputs
yolo_outputs = sorted(list((OUTPUTS_DIR / "det_yolo_v8n").glob("*.*")))
show_images_grid(yolo_outputs, cols=5)

<Figure size 1400x600 with 10 Axes>

In [8]:
# Load YOLOv8s Model (Detection Model 2)
yolo_model_s = YOLO("yolov8s.pt")
print("YOLOv8s model loaded")

YOLOv8s model loaded


In [9]:
# Run YOLOv8s Inference and Measure Time
import time

yolo_s_out_dir = OUTPUTS_DIR / "det_yolo_v8s"
yolo_s_out_dir.mkdir(exist_ok=True)

images = sorted(list(TEST_IMAGES_DIR.glob("*.*")))[:10]

times_s = []
for p in images:
    t0 = time.time()
    results = yolo_model_s.predict(source=str(p), conf=0.25, verbose=False)
    t1 = time.time()
    times_s.append(t1 - t0)

    annotated = results[0].plot()
    out_path = yolo_s_out_dir / p.name
    cv2.imwrite(str(out_path), annotated)

print("Saved YOLOv8s outputs to:", yolo_s_out_dir)
print("Avg inference time (sec/image):", round(float(np.mean(times_s)), 4))

Saved YOLOv8s outputs to: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 5\Task5-1_ObjectDetection_Segmentation\outputs\det_yolo_v8s
Avg inference time (sec/image): 0.2796


In [10]:
# Preview YOLOv8s Outputs
yolo_s_outputs = sorted(list((OUTPUTS_DIR / "det_yolo_v8s").glob("*.*")))
show_images_grid(yolo_s_outputs, cols=5)

<Figure size 1400x600 with 10 Axes>

In [11]:
# Imports for Faster R-CNN (Detection Model 3)
import torch
import torchvision
from torchvision.transforms import functional as F
from torchvision.models.detection import fasterrcnn_resnet50_fpn

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)

torch: 2.10.0+cpu
torchvision: 0.25.0+cpu


In [12]:
# Imports for Faster R-CNN (Detection Model 3)
import torch
import torchvision
from torchvision.transforms import functional as F
from torchvision.models.detection import fasterrcnn_resnet50_fpn

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)

torch: 2.10.0+cpu
torchvision: 0.25.0+cpu


In [13]:
# COCO Labels for Faster R-CNN
COCO_CLASSES = [
    "__background__", "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat",
    "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep",
    "cow", "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase",
    "frisbee", "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard", "tennis racket",
    "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange",
    "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch", "potted plant", "bed",
    "dining table", "toilet", "tv", "laptop", "mouse", "remote", "keyboard", "cell phone", "microwave", "oven",
    "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors", "teddy bear", "hair drier", "toothbrush"
]
print("COCO classes:", len(COCO_CLASSES))

COCO classes: 81


In [14]:
# Load Faster R-CNN Pretrained Model
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
frcnn = fasterrcnn_resnet50_fpn(weights="DEFAULT")
frcnn.to(device)
frcnn.eval()

print("Device:", device)
print("Faster R-CNN loaded")

Device: cpu
Faster R-CNN loaded


In [15]:
# Run Faster R-CNN on 10 Images and Save Outputs
import time

frcnn_out_dir = OUTPUTS_DIR / "det_fasterrcnn"
frcnn_out_dir.mkdir(exist_ok=True)

images = sorted(list(TEST_IMAGES_DIR.glob("*.*")))[:10]

def draw_boxes_cv2(bgr_img, boxes, labels, scores, score_thr=0.5):
    img = bgr_img.copy()
    for box, lab, sc in zip(boxes, labels, scores):
        if sc < score_thr:
            continue
        x1, y1, x2, y2 = box.astype(int)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        text = f"{lab} {sc:.2f}"
        cv2.putText(img, text, (x1, max(15, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return img

times_frcnn = []

with torch.no_grad():
    for p in images:
        # Load image
        pil_img = Image.open(p).convert("RGB")
        img_tensor = F.to_tensor(pil_img).to(device)

        # Inference
        t0 = time.time()
        out = frcnn([img_tensor])[0]
        t1 = time.time()
        times_frcnn.append(t1 - t0)

        boxes = out["boxes"].cpu().numpy()
        labels_idx = out["labels"].cpu().numpy()
        scores = out["scores"].cpu().numpy()
        labels = [COCO_CLASSES[i] if i < len(COCO_CLASSES) else str(i) for i in labels_idx]

        # Draw on original (cv2)
        bgr = cv2.imread(str(p))
        annotated = draw_boxes_cv2(bgr, boxes, labels, scores, score_thr=0.5)

        out_path = frcnn_out_dir / p.name
        cv2.imwrite(str(out_path), annotated)

print("Saved Faster R-CNN outputs to:", frcnn_out_dir)
print("Avg inference time (sec/image):", round(float(np.mean(times_frcnn)), 4))

Saved Faster R-CNN outputs to: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 5\Task5-1_ObjectDetection_Segmentation\outputs\det_fasterrcnn
Avg inference time (sec/image): 4.4654


In [16]:
# Preview Faster R-CNN Outputs
frcnn_outputs = sorted(list((OUTPUTS_DIR / "det_fasterrcnn").glob("*.*")))
show_images_grid(frcnn_outputs, cols=5)

<Figure size 1400x600 with 10 Axes>

In [17]:
# Load YOLOv8 Segmentation Model (Segmentation Model 1)
from ultralytics import YOLO

yolo_seg = YOLO("yolov8n-seg.pt")
print("YOLOv8n-seg loaded")

YOLOv8n-seg loaded


In [18]:
# Run YOLOv8 Segmentation on 10 Images and Save Outputs
import time
import numpy as np

yolo_seg_out_dir = OUTPUTS_DIR / "seg_yolo_v8n"
yolo_seg_out_dir.mkdir(exist_ok=True)

images = sorted(list(TEST_IMAGES_DIR.glob("*.*")))[:10]

times_seg = []
for p in images:
    t0 = time.time()
    results = yolo_seg.predict(source=str(p), conf=0.25, verbose=False)
    t1 = time.time()
    times_seg.append(t1 - t0)

    annotated = results[0].plot()  # includes masks overlay
    out_path = yolo_seg_out_dir / p.name
    cv2.imwrite(str(out_path), annotated)

print("Saved YOLO Seg outputs to:", yolo_seg_out_dir)
print("Avg inference time (sec/image):", round(float(np.mean(times_seg)), 4))

Saved YOLO Seg outputs to: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 5\Task5-1_ObjectDetection_Segmentation\outputs\seg_yolo_v8n
Avg inference time (sec/image): 0.2168


In [19]:
# Preview YOLOv8 Segmentation Outputs
seg_outputs = sorted(list((OUTPUTS_DIR / "seg_yolo_v8n").glob("*.*")))
show_images_grid(seg_outputs, cols=5)

<Figure size 1400x600 with 10 Axes>

In [20]:
# Imports for DeepLabV3 (Segmentation Model 2)
from torchvision.models.segmentation import deeplabv3_resnet50
from torchvision import transforms

print("DeepLabV3 import OK")

DeepLabV3 import OK


In [21]:
# Load DeepLabV3 Pretrained Model
deeplab = deeplabv3_resnet50(weights="DEFAULT")
deeplab.to(device)
deeplab.eval()

print("DeepLabV3 loaded on device:", device)

DeepLabV3 loaded on device: cpu


In [22]:
# Run DeepLabV3 Segmentation and Save Outputs
import numpy as np
import time

deeplab_out_dir = OUTPUTS_DIR / "seg_deeplabv3"
deeplab_out_dir.mkdir(exist_ok=True)

transform = transforms.Compose([
    transforms.ToTensor(),
])

times_dl = []

with torch.no_grad():
    for p in images:
        img = Image.open(p).convert("RGB")
        img_tensor = transform(img).unsqueeze(0).to(device)

        t0 = time.time()
        output = deeplab(img_tensor)["out"]
        t1 = time.time()
        times_dl.append(t1 - t0)

        mask = output.argmax(1).squeeze().cpu().numpy()

        # Normalize mask for visualization
        mask_vis = (mask * 10) % 255
        mask_vis = mask_vis.astype(np.uint8)

        mask_color = cv2.applyColorMap(mask_vis, cv2.COLORMAP_JET)

        bgr = cv2.imread(str(p))
        h, w = bgr.shape[:2]
        mask_color_resized = cv2.resize(mask_color, (w, h), interpolation=cv2.INTER_NEAREST)

        overlay = cv2.addWeighted(bgr, 0.6, mask_color_resized, 0.4, 0)

        out_path = deeplab_out_dir / p.name
        cv2.imwrite(str(out_path), overlay)

print("Saved DeepLabV3 outputs to:", deeplab_out_dir)
print("Avg inference time (sec/image):", round(float(np.mean(times_dl)), 4))

Saved DeepLabV3 outputs to: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 5\Task5-1_ObjectDetection_Segmentation\outputs\seg_deeplabv3
Avg inference time (sec/image): 11.506


In [23]:
# Preview DeepLabV3 Outputs
deeplab_outputs = sorted(list((OUTPUTS_DIR / "seg_deeplabv3").glob("*.*")))
show_images_grid(deeplab_outputs, cols=5)

<Figure size 1400x600 with 10 Axes>

In [24]:
# Build Simple U-Net Model (Segmentation Model 3)
import tensorflow as tf
from tensorflow.keras import layers, models

def build_unet(input_shape=(256, 256, 3)):
    inputs = layers.Input(input_shape)

    # Encoder
    c1 = layers.Conv2D(16, 3, activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(16, 3, activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D()(c1)

    c2 = layers.Conv2D(32, 3, activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(32, 3, activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D()(c2)

    # Bottleneck
    b1 = layers.Conv2D(64, 3, activation='relu', padding='same')(p2)

    # Decoder
    u1 = layers.UpSampling2D()(b1)
    u1 = layers.Concatenate()([u1, c2])
    c3 = layers.Conv2D(32, 3, activation='relu', padding='same')(u1)

    u2 = layers.UpSampling2D()(c3)
    u2 = layers.Concatenate()([u2, c1])
    c4 = layers.Conv2D(16, 3, activation='relu', padding='same')(u2)

    outputs = layers.Conv2D(1, 1, activation='sigmoid')(c4)

    return models.Model(inputs, outputs)

unet = build_unet()
unet.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 256,  │        448 │ input_layer[0][0] │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 256, 256,  │      2,320 │ conv2d[0][0]      │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 128,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 128, 128,  │      4,640 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 128, 128,  │      9,248 │ conv2d_2[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 64,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 64,    │     18,496 │ max_pooling2d_1[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d       │ (None, 128, 128,  │          0 │ conv2d_4[0][0]    │
│ (UpSampling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128, 128,  │          0 │ up_sampling2d[0]… │
│ (Concatenate)       │ 96)               │            │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 128, 128,  │     27,680 │ concatenate[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_1     │ (None, 256, 256,  │          0 │ conv2d_5[0][0]    │
│ (UpSampling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 256, 256,  │          0 │ up_sampling2d_1[… │
│ (Concatenate)       │ 48)               │            │ conv2d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 256, 256,  │      6,928 │ concatenate_1[0]… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 256, 256,  │         17 │ conv2d_6[0][0]    │
│                     │ 1)                │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 69,777 (272.57 KB)

 Trainable params: 69,777 (272.57 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
# Run U-Net Inference (Random Weights Demo)
unet_out_dir = OUTPUTS_DIR / "seg_unet"
unet_out_dir.mkdir(exist_ok=True)

import time

times_unet = []

for p in images:
    img = cv2.imread(str(p))
    img_resized = cv2.resize(img, (256, 256))
    img_norm = img_resized / 255.0
    img_input = np.expand_dims(img_norm, axis=0)

    t0 = time.time()
    pred = unet.predict(img_input, verbose=0)[0]
    t1 = time.time()
    times_unet.append(t1 - t0)

    mask = (pred.squeeze() * 255).astype(np.uint8)
    mask_color = cv2.applyColorMap(mask, cv2.COLORMAP_JET)

    mask_color_resized = cv2.resize(mask_color, (img.shape[1], img.shape[0]))
    overlay = cv2.addWeighted(img, 0.6, mask_color_resized, 0.4, 0)

    out_path = unet_out_dir / p.name
    cv2.imwrite(str(out_path), overlay)

print("Saved U-Net outputs to:", unet_out_dir)
print("Avg inference time (sec/image):", round(float(np.mean(times_unet)), 4))

Saved U-Net outputs to: E:\JYU\Deep-Learning for Cognitive Computing for Developers\Task 5\Task5-1_ObjectDetection_Segmentation\outputs\seg_unet
Avg inference time (sec/image): 0.2412


In [26]:
# Preview U-Net Outputs
unet_outputs = sorted(list((OUTPUTS_DIR / "seg_unet").glob("*.*")))
show_images_grid(unet_outputs, cols=5)

<Figure size 1400x600 with 10 Axes>

In [27]:
# Create Detection and Segmentation Performance Summary Table
import pandas as pd

summary_data = [
    ["YOLOv8n", "Detection", 0.1578, "One-stage detector"],
    ["YOLOv8s", "Detection", 0.3414, "One-stage (larger backbone)"],
    ["Faster R-CNN", "Detection", 4.1923, "Two-stage detector"],
    ["YOLOv8n-seg", "Segmentation", 0.1908, "Instance segmentation"],
    ["DeepLabV3", "Segmentation", 10.9637, "Semantic segmentation"],
    ["U-Net (random)", "Segmentation", 0.1875, "Encoder-decoder (untrained demo)"],
]

df_summary = pd.DataFrame(summary_data, columns=[
    "Model", "Task Type", "Avg Inference Time (sec/image)", "Architecture Type"
])

df_summary

,Model,Task Type,Avg Inference Time (sec/image),Architecture Type
0,YOLOv8n,Detection,0.1578,One-stage detector
1,YOLOv8s,Detection,0.3414,One-stage (larger backbone)
2,Faster R-CNN,Detection,4.1923,Two-stage detector
3,YOLOv8n-seg,Segmentation,0.1908,Instance segmentation
4,DeepLabV3,Segmentation,10.9637,Semantic segmentation
5,U-Net (random),Segmentation,0.1875,Encoder-decoder (untrained demo)


## mAP (Mean Average Precision) – Theoretical Explanation

In object detection, performance is commonly evaluated using Mean Average Precision (mAP).

The evaluation is based on:

Intersection over Union (IoU):
Measures the overlap between predicted bounding box and ground truth box.

Precision–Recall Curve:
Precision is computed at different confidence thresholds.

Average Precision (AP):
Area under the precision–recall curve for one class.

Mean Average Precision (mAP):
Mean of AP values across all classes.

Two common metrics are:

mAP@0.5 → IoU threshold = 0.5

mAP@0.5:0.95 → Average across IoU thresholds from 0.5 to 0.95

In this task, pretrained models were used for inference on custom test images.
Therefore, comparison was performed based on:

Qualitative visual results

Architectural differences

Average inference time per image

Official benchmark mAP values reported by the original model implementations (COCO dataset) are referenced in the report section.

## Model Implementation References

The following official implementations were used:

YOLOv8 (Detection & Segmentation)
Ultralytics Implementation
https://github.com/ultralytics/ultralytics

Faster R-CNN (ResNet50-FPN)
Torchvision Detection Models
https://pytorch.org/vision/stable/models.html

DeepLabV3 (ResNet50 Backbone)
Torchvision Segmentation Models
https://pytorch.org/vision/stable/models.html

U-Net (Custom Keras Implementation)
Built manually following encoder–decoder architecture described in literature.